# Direct Preference Optimization (DPO) for LLM Alignment

# Step 0: Prerequisite

In [ ]:
!pip install -qq llms-from-scratch

In [ ]:
from importlib.metadata import version

pkgs = [
    "tiktoken",    # Tokenizer
    "torch",       # Deep learning library
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

&nbsp;
# Step 1: Preparing a preference dataset for DPO

Recall that we will need a **Preference dataset** to align our LLM using DPO. Here we will work with a dataset of size 1100 that contains more polite (preferred) and less polite (rejected) responses.

#### FAQ: How does a Preference dataset look like?
#### A: Imagine the dataset takes the form ($x$, $y_w$, $y_l$), where $x$ represents the prompt, $y_w$ represents the preferred response, and $y_l$ represents the rejected response.

## Step 1.1: Loading dataset

In [ ]:
import json
import os
import requests

def load_file(file_path):
  with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()

  data = json.loads(text_data)
  return data

# Make sure you have uploaded the file to current working directory
file_path = "instruction-data-with-preference.json"

data = load_file(file_path)
print("Number of entries:", len(data))

Let's take a look at two example entries:

In [ ]:
import pprint

pprint.pp(data[50])

In [ ]:
pprint.pp(data[999])

As we can see above, the dataset consists of 5 keys:
- The `'instruction'` and `'input'` that are used as LLM inputs
- The `'output'` contains the response the model (assume it has been trained on via the instruction finetuning step)
- the `'chosen'` and `'rejected'` entries are the entries we use for DPO; here `'chosen'` is the preferred response, and `'rejected'` is the dispreferred response

&nbsp;   
The goal is to get the model to follow the style of the chosen over the rejected responses.

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [ ]:
model_input = format_input(data[50])
print(model_input)

- Similarly, we can format the chosen and rejected responses using the Alpaca prompt style:

In [ ]:
desired_response = f"### Response:\n{data[50]['chosen']}"
print(desired_response)

In [ ]:
possible_response = f"### Response:\n{data[50]['rejected']}"
print(possible_response)

## Step 1.2: Creating training, validation, and test splits

- Next, we divide the dataset into 3 subsets, 85% training data, 5% validation data, and 10% test data:

In [ ]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [ ]:
print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

## Step 1.3: Developing a `PreferenceDataset` class & batch processing function

In [ ]:
import torch
from torch.utils.data import Dataset


class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            # for each (x, y_w, y_l), we format + encode them
            prompt = format_input(entry)
            rejected_response = entry["rejected"]
            chosen_response = entry["chosen"]

            prompt_tokens = tokenizer.encode(prompt)
            chosen_full_text = f"{prompt}\n\n### Response:\n{chosen_response}"
            rejected_full_text = f"{prompt}\n\n### Response:\n{rejected_response}"
            chosen_full_tokens = tokenizer.encode(chosen_full_text)
            rejected_full_tokens = tokenizer.encode(rejected_full_text)

            self.encoded_texts.append({
                "prompt": prompt_tokens,
                "chosen": chosen_full_tokens,
                "rejected": rejected_full_tokens,
            })

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


- Along with an updated `PreferenceDataset` class, we also need an updated batch collation function that we use to pad the sequences in each batch to an equal length so that we can assemble them in batches
- I added comments to the code below to illustrate the process; however, it might be easiest to understand how it works by looking at the example inputs and outputs further below:

In [ ]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    allowed_max_length=None,
    mask_prompt_tokens=True,
    device="cpu"
):
    # Initialize lists to hold batch data
    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "rejected_mask": [],
        "chosen_mask": []

    }

    # Determine the longest sequence to set a common padding length
    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max(len(item[key])+1 for item in batch)
            max_length_common = max(max_length_common, current_max)

    # Process each item in the batch
    for item in batch:
        prompt = torch.tensor(item["prompt"])
        batch_data["prompt"].append(prompt)

        for key in ["chosen", "rejected"]:
            # Adjust padding according to the common maximum length
            sequence = item[key]
            padded = sequence + [pad_token_id] * (max_length_common - len(sequence))
            mask = torch.ones(len(padded)).bool()

            # Set mask for all padding tokens to False
            mask[len(sequence):] = False

            # Set mask for all input tokens to False
            # +2 sets the 2 newline ("\n") tokens before "### Response" to False
            if mask_prompt_tokens:
                mask[:prompt.shape[0]+2] = False

            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key}_mask"].append(mask)

    # Final processing
    for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        # Stack all sequences into a tensor for the given key
        tensor_stack = torch.stack(batch_data[key])

        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]

        # Move to the specified device
        batch_data[key] = tensor_stack.to(device)

    return batch_data

If you are meticulous, you might notice that the prompt is included in the input sequence but later masked during loss computation. The reason is that it is not meaningful to evaluate the probability of a response without context. Instead, we want the model to learn the probability of generating the response conditioned on a given prompt. However, since the prompt itself is part of the provided context rather than something the model should learn to generate, we mask the prompt tokens when computing the loss so that only the response tokens contribute to training.

&nbsp;

Transformers compute each token based on previous tokens. For example:
- Prompt: "Explain gravity simply."
- Response: "Gravity is a force ..."

The probability of "Gravity" depends on previous tokens, i.e., `P(Gravity | "Explain gravity simply.")`, so it is crucial to keep the prompt tokens.

&nbsp;

## [Optional]: Simple example of Batching

In [ ]:
from functools import partial

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Think of this as a function with some pre-filled arguments
customized_collate_fn = partial(
    custom_collate_fn,        # Our original function (defined above)

    # Pre-filled arguments #
    device=device,            # Put the data directly on a GPU if available
    mask_prompt_tokens=True,  # This is optional
    allowed_max_length=1024   # The supported context length of the model
)

In [ ]:
# Here we take 2 entries as example
example_data = data[:2]

for i in example_data:
    print()
    pprint.pp(i)

In [ ]:
import tiktoken
from torch.utils.data import DataLoader

# Instantiate the tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Create the dataset that takes the desired "pattern" (x, y_w, y_l)
example_dataset = PreferenceDataset(example_data, tokenizer)

In the following code:
- `DataLoader` "collects" a list of samples based on `batch_size`
- `collate_fn` will take in the list of samples -> Preprocess (e.g., padding) + Transform into desired format

In [ ]:
example_dataloader = DataLoader(
    example_dataset,
    batch_size=2,
    collate_fn=customized_collate_fn,
    shuffle=False
)

In [ ]:
# Let's take the first batch and visualize
batch = next(iter(example_dataloader))

print("batch.keys:", batch.keys())

In [ ]:
# We have batch_size=2, so we see two lists of prompt tokens
batch["prompt"]

In [ ]:
# Notice how the first few tokens are from "prompt" and the last few are padding tokens|
batch["chosen"]

In [ ]:
def decode_tokens_from_batch(token_ids, tokenizer):
    '''Decipher tokens back to texts for easier inspection'''
    ids_in_python_list = token_ids.flatten().tolist()
    return tokenizer.decode(ids_in_python_list)

In [ ]:
# Polite version
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0], # [0] for the first entry in the batch
    tokenizer=tokenizer,
)

# Impolite version

# text = decode_tokens_from_batch(
#     token_ids=batch["rejected"][0],
#     tokenizer=tokenizer,
# )

print(text) # Notice that we have <|endoftext|>, but they will be ignored when computing loss

Next, let's talk about the data masks, which will be used to ignore prompt and padding tokens when computing the DPO loss later.

In [ ]:
# Note that the mask tensor has the same size as the input tensor
# Just that the contents of the masks are boolean values
print("chosen inputs:", batch["chosen"][0].shape)
print("chosen mask:  ", batch["chosen_mask"][0].shape)

In [ ]:
batch["chosen_mask"][0]

- The `True` values denote token IDs that correspond to the actual response
- the `False` tokens correspond to token IDs that correspond to either
  - **prompt tokens** (if we set `mask_prompt_tokens=True` in the `customized_collate_fn` function, which we previously did) or;
  - **padding tokens**
- Hence, we can use the mask as a selection mask to select only the token IDs that correspond to the response, that is, stripping all prompt and padding tokens, as we can see below:

In [ ]:
# Polite version response-only
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0][batch["chosen_mask"][0]],
    tokenizer=tokenizer,
)

# Impolite version response-only
# text = decode_tokens_from_batch(
#     token_ids=batch["rejected"][0][batch["rejected_mask"][0]],
#     tokenizer=tokenizer,
# )

print(text)

## End of [Optional]: Simple Example of Batching

&nbsp;

&nbsp;
## Step 1.4 Creating training, validation, and test set data loaders

Above, we worked with a small example subsets from the preference dataset for illustration purposes. Let's now create the actual training, validation, and test set data loaders.

In [ ]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = PreferenceDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,   # prevent the model from learning the order of data + generalization
    drop_last=True, # for stable TRAINING (due to the use of batch stats), discard last batch (fewer samples)
    num_workers=num_workers
)

In [ ]:
val_dataset = PreferenceDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False, # want to evaluate everything, no need to drop
    num_workers=num_workers
)

test_dataset = PreferenceDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False, # want to evaluate everything, no need to drop
    num_workers=num_workers
)

- Let's iterate through the data loader and take a look at the dataset shapes:

In [ ]:
# Visualize the shape of the "chosen" and "rejected" entries in first 5 batch

# Note that each row has a different shape
# because it would be inefficient to pad all samples to the longest sample of the dataset
print("Train loader:")
for i, batch in enumerate(train_loader):
    # stop after 5 batches
    if i >= 5:
        break

    print(
        batch["chosen"].shape,
        batch["rejected"].shape,
    )

&nbsp;
# Step 2: Loading a finetuned LLM for DPO alignment

LLM alignment steps, such as RLHF or DPO, assume that we already have an instruction-finetuned model. We will use `gpt2-medium355M-sft.pth` uploaded onto HuggingFace Hub.

Note that in DPO, we will work with two LLMs:
- Policy model (the LLM that we want to optimize); and
- Reference model (the original model that we keep unchanged)


Thus, we will name the **model to be fine-tuned** as `policy_model` and instantiate a second one where we refer to as the `reference_model`.

In [ ]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="JasonOng0109/gpt2-medium355M-sft",
    filename="model.pth"
)

In [ ]:
from llms_from_scratch.ch04 import GPTModel

In [ ]:
BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "emb_dim": 1024,         # Embedding dimension
    "n_layers": 24,          # Number of transformer layers
    "n_heads": 16,           # Number of attention heads
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

In [ ]:
# Create a GPT Model of the above configuration (no weights yet)
policy_model = GPTModel(BASE_CONFIG)

# Load the model with trained weights
policy_model.load_state_dict(
    torch.load(
        model_path,
        map_location=torch.device("cpu"),
        weights_only=True
    )
)
policy_model.to(device)
policy_model.eval();

In [ ]:
# Repeat for reference model instantiation
reference_model = GPTModel(BASE_CONFIG)
reference_model.load_state_dict(
    torch.load(
        model_path,
        map_location=torch.device("cpu"),
        weights_only=True
    )
)
reference_model.to(device)
reference_model.eval();

&nbsp;
# Step 3: DPO Loss Function

Note that the DPO loss code below is based on the method proposed in the [Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290) paper


For reference, the core DPO equation is shown again below:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/dpo/3.webp?123" width=800px>

In code, we can implement the DPO loss as follows:

In [ ]:
import torch.nn.functional as F

def compute_dpo_loss(
      model_chosen_logprobs,        # log π_θ(y_w|x)
      model_rejected_logprobs,      # log π_θ(y_l|x)
      reference_chosen_logprobs,    # log π_ref(y_w|x)
      reference_rejected_logprobs,  # log π_ref(y_l|x)
      beta=0.1,
    ):
    """Compute the DPO loss for a batch of policy and reference model log probabilities.

    Args:
        model_chosen_logprobs: Log probabilities of the policy model for the chosen responses. Shape: (batch_size,)
        model_rejected_logprobs: Log probabilities of the policy model for the rejected responses. Shape: (batch_size,)
        reference_chosen_logprobs: Log probabilities of the reference model for the chosen responses. Shape: (batch_size,)
        reference_rejected_logprobs: Log probabilities of the reference model for the rejected responses. Shape: (batch_size,)
        beta: Temperature parameter for the DPO loss; typically something in the range of 0.1 to 0.5. We ignore the reference model as beta -> 0.

    Returns:
        A tuple of three tensors: (loss, chosen_rewards, rejected_rewards).
    """

    # Calculate the difference in log probabilities (logits) for the chosen and rejected samples
    # for policy model and reference model respectively
    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs

    # Using property of log to achieve the proposed equation
    logits = model_logratios - reference_logratios

    # DPO (Eq. 7 of https://arxiv.org/pdf/2305.18290.pdf)
    losses = -F.logsigmoid(beta * logits)

    # Optional values to track progress during training
    chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()

    # .mean() to average over the samples in the batch
    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()

So far, we assumed that the log probabilities were already computed. Let's now define a `compute_logprobs` function that we can use to compute these log probabilities that were passed into the `compute_dpo_loss` function above, that is, the values $\pi_\theta (y_w \mid x)$, ${\pi_\theta (y_l \mid x)}$, and so forth:

In [ ]:
def compute_logprobs(logits, labels, selection_mask=None):
    """
    Compute log probabilities.

    Args:
      logits: Tensor of shape (batch_size, num_tokens, vocab_size); vocab_size scores for each token in each sequence.
      labels: Tensor of shape (batch_size, num_tokens); true token IDs
      selection_mask: Tensor for shape (batch_size, num_tokens); recall we use mask to select actual response only

    Returns:
      mean_log_prob: Mean log probability excluding padding tokens.
    """

    # Labels are the inputs shifted by one -> because language models predict the next token.
    # Hence we want to remove the first token (usually BOS aka Beginning of Sentence).
    labels = labels[:, 1:].clone()

    # Truncate logits to match the labels num_tokens -> last prediction has no label
    logits = logits[:, :-1, :]

    # The two steps meant to align (intuitively)
    # labels = [BOS, I, love, pizza]          -> [I, love, pizza]
    # logits = [I, love, pizza, <next_token>] -> [I, love, pizza]

    log_probs = F.log_softmax(logits, dim=-1) # log probability is more stable

    # High-level idea is to choose the log probability of the correct token
    # From the 3D log_probs, for every token in each sequence,
    # we want to get only the log prob value for the "correct" one (based on the token IDs in true label at each token position)
    selected_log_probs = torch.gather(
        input=log_probs,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    # After that, log_probs no longer need to carry 50257 dimensions for each token position
    # shape (batch_size, num_tokens-1)

    if selection_mask is not None:
        mask = selection_mask[:, 1:].clone()

        # Apply the mask to filter out padding tokens
        selected_log_probs = selected_log_probs * mask

        # Calculate the average log probability excluding padding tokens
        # This averages over the tokens, so the shape is (batch_size,)
        avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)

        return avg_log_prob

    else:
        return selected_log_probs.mean(-1)

In [ ]:
def compute_dpo_loss_batch(batch, policy_model, reference_model, beta):
    """Compute the DPO loss on an input batch"""

    policy_chosen_log_probas = compute_logprobs(
        logits=policy_model(batch["chosen"]),   # logits output from the policy model
        labels=batch["chosen"],
        selection_mask=batch["chosen_mask"]
    )
    policy_rejected_log_probas = compute_logprobs(
        logits=policy_model(batch["rejected"]), # logits output from the policy model
        labels=batch["rejected"],
        selection_mask=batch["rejected_mask"]
    )

    # reference model is not trained, and this prevents PyTorch from building an unnecessary gradient graph -> save memory
    with torch.no_grad():
        ref_chosen_log_probas = compute_logprobs(
            logits=reference_model(batch["chosen"]),
            labels=batch["chosen"],
            selection_mask=batch["chosen_mask"]
        )
        ref_rejected_log_probas = compute_logprobs(
            logits=reference_model(batch["rejected"]),
            labels=batch["rejected"],
            selection_mask=batch["rejected_mask"]
        )
    loss, chosen_rewards, rejected_rewards = compute_dpo_loss(
        model_chosen_logprobs=policy_chosen_log_probas,
        model_rejected_logprobs=policy_rejected_log_probas,
        reference_chosen_logprobs=ref_chosen_log_probas,
        reference_rejected_logprobs=ref_rejected_log_probas,
        beta=beta
    )
    return loss, chosen_rewards, rejected_rewards

`compute_dpo_loss_batch` calculates loss only for one batch of data. Here we extend this function to work for a specified `num_batches` in a data loader:

In [ ]:
def compute_dpo_loss_loader(data_loader, policy_model, reference_model, beta, num_batches=None):
    """Apply compute_dpo_loss_batch to a whole data loader"""

    total_loss, total_chosen_rewards, total_rejected_rewards = 0., 0., 0.
    if len(data_loader) == 0:
        return float("nan")

    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, batch in enumerate(data_loader):
        if i < num_batches:
            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(
                batch=batch,
                policy_model=policy_model,
                reference_model=reference_model,
                beta=beta
            )
            total_loss += loss.item()
            total_chosen_rewards += chosen_rewards.item()
            total_rejected_rewards += rejected_rewards.item()

        else:
            break

    # calculate average
    total_loss /= num_batches
    total_chosen_rewards /= num_batches
    total_rejected_rewards /= num_batches
    return total_loss, total_chosen_rewards, total_rejected_rewards

Lastly, we define a convenience function for training, where it computes the DPO loss and rewards for both the training and validation loader for **logging purposes**:

In [ ]:
def evaluate_dpo_loss_loader(policy_model, reference_model, train_loader, val_loader, beta, eval_iter):
    """Compute the DPO loss for the training and validation dataset for logging"""

    policy_model.eval()
    with torch.no_grad():
        train_loss, train_chosen_rewards, train_rejected_rewards = compute_dpo_loss_loader(
            data_loader=train_loader,
            policy_model=policy_model,
            reference_model=reference_model,
            beta=beta,
            num_batches=eval_iter
        )

        val_loss, val_chosen_rewards, val_rejected_rewards = compute_dpo_loss_loader(
            data_loader=val_loader,
            policy_model=policy_model,
            reference_model=reference_model,
            beta=beta,
            num_batches=eval_iter
        )

    res = {
        "train_loss": train_loss,
        "train_chosen_reward": train_chosen_rewards,
        "train_rejected_reward": train_rejected_rewards,
        "val_loss": val_loss,
        "val_chosen_reward": val_chosen_rewards,
        "val_rejected_reward": val_rejected_rewards
    }

    policy_model.train()
    return res

In summary, the flow is:

compute `logits` via the models $\rightarrow$ `compute_logprobs` from logits $\rightarrow$ compute `compute_dpo_loss` from log probabilities

We also have the following functions, where:
1. `compute_dpo_loss_batch` utilize `compute_dpo_loss` to calculate the loss for one batch of data
2. `compute_dpo_loss_loader` utilize `compute_dpo_loss_batch` to repeat the DPO loss calculation for each batch
3. `evaluate_dpo_loss_loader` is an optional function that applies the `compute_dpo_loss_batch` to both the training and validation set data loaders for logging purposes

&nbsp;
# Step 4: Training the model

Before we start the training, let's check out the initial losses and rewards by using `evaluate_dpo_loss_loader`. Notice how the `Training loss` and `Validation loss` are the same because they are the same model before we start the alignment training.

In [ ]:
torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader

res = evaluate_dpo_loss_loader(
    policy_model=policy_model,
    reference_model=reference_model,
    train_loader=train_loader,
    val_loader=val_loader,
    beta=0.1,
    eval_iter=5
)

print("Training loss:", res["train_loss"])
print("Validation loss:", res["val_loss"])

print("Train reward margin:", res["train_chosen_reward"] - res["train_rejected_reward"])
print("Val reward margin:", res["val_chosen_reward"] - res["val_rejected_reward"])

Let's take a look at the first 3 examples of the initial model responses on the validation set:

In [ ]:
# Utility functions
from llms_from_scratch.ch05 import (
    generate,          # Generate response tokens
    text_to_token_ids, # Utility to convert text to token
    token_ids_to_text  # Utility to convert token to text
)

In [ ]:
torch.manual_seed(123)

for entry in val_data[:3]:
    # Recall how we format the input earlier
    input_text = format_input(entry)

    # Use (untrained) policy model to generate response to the input
    token_ids = generate(
        model=policy_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )

    # Decode the token into text
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
)

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print("\n-------------------------------------\n")

In [ ]:
def train_model_dpo_simple(
    policy_model, reference_model, train_loader, val_loader,
    optimizer, num_epochs, beta,
    eval_freq, eval_iter, tokenizer
):

    # Initialize lists to track losses and tokens seen
    tracking = {
        "train_losses": [],
        "train_chosen_rewards": [],
        "train_rejected_rewards": [],
        "val_losses": [],
        "val_chosen_rewards": [],
        "val_rejected_rewards": [],
        "tokens_seen": []
    }
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        policy_model.train()  # Set model to training mode

        for batch in train_loader:

            optimizer.zero_grad()  # Reset loss gradients from previous batch iteration

            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(
                batch=batch,
                policy_model=policy_model,
                reference_model=reference_model,
                beta=beta
            )

            loss.backward()  # Calculate loss gradients
            optimizer.step()  # Update model weights using loss gradients

            tokens_seen += batch["chosen"].numel()
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                res = evaluate_dpo_loss_loader(
                    policy_model=policy_model,
                    reference_model=reference_model,
                    train_loader=train_loader,
                    val_loader=val_loader,
                    beta=beta,
                    eval_iter=eval_iter
                )
                tracking["train_losses"].append(res["train_loss"])
                tracking["train_chosen_rewards"].append(res["train_chosen_reward"])
                tracking["train_rejected_rewards"].append(res["train_rejected_reward"])
                tracking["val_losses"].append(res["val_loss"])
                tracking["val_chosen_rewards"].append(res["val_chosen_reward"])
                tracking["val_rejected_rewards"].append(res["val_rejected_reward"])
                tracking["tokens_seen"].append(tokens_seen)
                train_reward_margin = res["train_chosen_reward"] - res["train_rejected_reward"]
                val_reward_margin = res["val_chosen_reward"] - res["val_rejected_reward"]

                print(
                    f"Ep {epoch+1} (Step {global_step:06d}): "
                    f"Train loss {res['train_loss']:.3f}, Val loss {res['val_loss']:.3f}, "
                    f"Train reward margins {train_reward_margin:.3f}, "
                    f"Val reward margins {val_reward_margin:.3f}"
                )

    return tracking

Before we execute the following code cell that starts the training, here are a few notes about some of the settings:
 - Only pass in the parameters of the policy model into the `AdamW` optimizer because we don't want to modify the reference model
 - Only train for 1 epoch because DPO is very prone to collapse (the loss might improve, but the model will start generating nonsensical texts)
 - Use very small learning rate for DPO

In [ ]:
import time

start_time = time.time()

torch.manual_seed(123)


optimizer = torch.optim.AdamW(policy_model.parameters(), lr=5e-6, weight_decay=0.01)

num_epochs = 1
tracking = train_model_dpo_simple(
    policy_model=policy_model,
    reference_model=reference_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    num_epochs=num_epochs,
    beta=0.1, # value between 0.1 and 0.5; smaller value means more conservative update
    eval_freq=5,
    eval_iter=5,
    tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

As we can see based on the tracked results above, the loss improves. Also, the reward margins, which is the difference between the rewards of the chosen and the rejected responses, improve, which is a good sign.

&nbsp;
# Step 5: Analyzing the results

We can analyze the results by plotting the DPO loss:

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses, label="loss"):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_losses, label=f"Training {label}")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label=f"Validation {label}")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # only show integer labels on x-axis

    # Create a second x-axis for tokens seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(tokens_seen, train_losses, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig(f"{label}-plot.pdf")
    plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(tracking["train_losses"]))
plot_losses(
    epochs_seen=epochs_tensor,
    tokens_seen=tracking["tokens_seen"],
    train_losses=tracking["train_losses"],
    val_losses=tracking["val_losses"],
    label="loss",
)


DPO loss continues to improve, which is a good sign, but one should be cautious on training the model a bit further. Recall that DPO is prone to collapse and the model may start generating nonsensical responses.

In [ ]:
train_reward_margins = [i-j for i,j in zip(tracking["train_chosen_rewards"], tracking["train_rejected_rewards"])]
val_reward_margins = [i-j for i,j in zip(tracking["val_chosen_rewards"], tracking["val_rejected_rewards"])]

plot_losses(
    epochs_seen=epochs_tensor,
    tokens_seen=tracking["tokens_seen"],
    train_losses=train_reward_margins,
    val_losses=val_reward_margins,
    label="reward margins"
)

The reward margins improve and this mirrors the loss curve (hence a good sign).
While DPO losses and reward margins are valuable metrics to track during training, they might not be able to tell the whole story.


Lastly, and most importantly, we should conduct a **qualitative** check of the responses.

In [ ]:
torch.manual_seed(123)

for entry in val_data[:3]:

    input_text = format_input(entry)

    # Response from original model
    token_ids = generate(
        model=reference_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    reference_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    # Response from optimized model
    token_ids = generate(
        model=policy_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    policy_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nReference model response:\n>> {reference_response_text.strip()}")
    print(f"\nPolicy model response:\n>> {policy_response_text.strip()}")
    print("\n-------------------------------------\n")

From the reference model and policy model responses above, we observed that the optimized model (i.e., the policy model) indeed slightly changed its style compared to the original model (i.e., reference model)

For instance, `"Dance" can be classified as a verb.` changed to `The input string "Dance" could be classified as a verb.` which is a slightly more polite response (the use of "could" instead of "can" makes the statement sound less assertive and more tentative)

In [ ]:
# Test set evaluation
torch.manual_seed(123)

for entry in test_data[:3]:

    input_text = format_input(entry)

    # Response from original model
    token_ids = generate(
        model=reference_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    reference_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    # Response from optimized model
    token_ids = generate(
        model=policy_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    policy_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nReference model response:\n>> {reference_response_text.strip()}")
    print(f"\nPolicy model response:\n>> {policy_response_text.strip()}")
    print("\n-------------------------------------\n")

Again, we observed really polite tone from the policy model.

For instance, `The author of 'Pride and Prejudice' is Jane Austen.` changed to ` The author of 'Pride and Prejudice' would be Sir Walter Scott.` which is a slightly more polite response (due to the use of "would")

# Conclusion

This tutorial walked through how to build a DPO trainer from scratch to help illustrate the core ideas behind the method. In practice, however, there are well-maintained libraries that handle much of this complexity for you. For example, the Hugging Face TRL repository (https://github.com/huggingface/trl) provides a ready-to-use implementation of DPO training. With these tools, you can often get started by simply configuring a few arguments rather than implementing the entire training pipeline yourself. If you found this tutorial interesting and would like to experiment further, exploring these libraries is a great next step.